In [28]:
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama
from typing import TypedDict 

In [29]:
model=ChatOllama(model="llama3.2")

In [30]:
class Blogstatereview(TypedDict):
    title:str
    outline:str
    blog:str
    evalute:int


In [31]:
def outline(state : Blogstatereview)->Blogstatereview:
    title=state['title']
    prompt=f"Generate a detailed outline for a log on the topic - {title}"
    outline=model.invoke(prompt)
    state['outline']=outline
    return state

In [32]:
def blog(state:Blogstatereview)->Blogstatereview:
    title=state['title']
    outline=state['outline']
    prompt=f'Write a detailed blog on the tittle {title} using the following outline \n {outline}'
    blog=model.invoke(prompt).content
    state['blog']=blog
    return state

In [33]:
def evalution(state):
    # 1. Grab whatever inputs you need from state
    blog_content = state.get('blog')
    
    # 2. Invoke your model to review the content
    # Assuming 'model' or 'llm' is your ChatOllama instance
    response = model.invoke(f"Review this blog: {blog_content}") 
    
    # 3. Save it back to state (YOUR FIX IS HERE)
    # Use string quotes for the key, and extract '.content' from the message
    state['evalution'] = response.content  
    
    return state



In [34]:
graph=StateGraph(Blogstatereview)
graph.add_node("outline",outline)
graph.add_node("blog",blog)
graph.add_node("evalution",evalution)
graph.add_edge(START,'outline')
graph.add_edge('outline','blog')
graph.add_edge('blog','evalution')
graph.add_edge('evalution',END)

workflow=graph.compile()

In [35]:
input_state={'title':"Rise of AI in India"}
output_state=workflow.invoke(input_state)
print(output_state)

{'title': 'Rise of AI in India', 'outline': AIMessage(content='Here is a detailed outline for a log on "The Rise of AI in India":\n\n**I. Introduction**\n\n* Brief overview of Artificial Intelligence (AI) and its significance\n* Importance of AI in the Indian economy and society\n* Thesis statement: The rise of AI in India has brought about significant changes and opportunities, but also poses challenges that need to be addressed.\n\n**II. History of AI in India**\n\n* Early attempts at AI research in India (1960s-1980s)\n* Growth of AI research centers and universities in India\n* Government initiatives to promote AI development and adoption\n\n**III. Current State of AI in India**\n\n* Overview of the current AI landscape in India, including:\n + AI-powered startups and companies\n + AI-driven innovation and entrepreneurship\n + Adoption of AI in various industries (e.g., healthcare, finance, education)\n* Statistics on AI adoption rates in India (e.g., percentage of businesses using

In [36]:
import pprint

input_state = {'title': "Rise of AI in India"}
output_state = workflow.invoke(input_state)

print("🎯 FINAL GRAPH STATE:")
print("=" * 40)
pprint.pprint(output_state)


🎯 FINAL GRAPH STATE:
{'blog': 'The Rise of AI in India: Transforming the Economy, Society, and '
         'Industries\n'
         '\n'
         "India's journey towards becoming a global hub for Artificial "
         'Intelligence (AI) has been nothing short of remarkable. With its '
         'rich history of IT development and outsourcing services, India is '
         'now poised to take the lead in AI adoption, transforming its '
         'economy, society, and industries in the process. In this blog, we '
         'will delve into the historical background, current state, government '
         'initiatives, economic and societal implications, challenges, and '
         'opportunities surrounding the rise of AI in India.\n'
         '\n'
         'Historical Background\n'
         '\n'
         "India's IT industry has a rich history dating back to the 1980s, "
         "when it began as a small startup in Mumbai. The country's early "
         'successes were largely driven by the o